# Cápsula Docente: Introducción a RDD y DataFrames en Apache Spark

## Gestión de Datos y Tecnologías Big Data - Semana 2

---

Este notebook ejemplifica los conceptos fundamentales de **RDD (Resilient Distributed Datasets)** y **DataFrames** en Apache Spark, basándose en la presentación de la cápsula docente.



**Objetivos de Aprendizaje:**

- Comprender la abstracción fundamental de RDD en Spark

- Diferenciar entre transformaciones y acciones

- Aplicar operaciones comunes sobre RDDs

- Entender la evaluación perezosa (lazy evaluation)

- Introducirse en el uso de DataFrames

# 1. SETUP INICIAL

## Configuración del Entorno de Trabajo



Antes de comenzar a trabajar con Spark, necesitamos configurar nuestro entorno en Google Colab. Este proceso incluye:

1. Montar Google Drive para acceder a archivos

2. Instalar Apache Spark y sus dependencias

3. Configurar variables de entorno

4. Inicializar SparkSession y SparkContext

### 1.1 Montar Google Drive

Conectamos nuestro entorno de Google Colab con Google Drive para acceder a los archivos necesarios.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 Instalar Apache Spark y Dependencias

Instalamos Java (JDK 8), Apache Spark 3.5.9, PySpark y findspark. Siempre es importante verificar la Versión disponible en https://downloads.apache.org/spark/  , debido que se van actualizando constantemente.

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz
!tar xf spark-3.5.9-bin-hadoop3.tgz
!pip install -q pyspark > /dev/null
!pip install -q findspark > /dev/null
print('✓ Apache Spark 3.5.9 instalado correctamente')

✓ Apache Spark 3.5.9 instalado correctamente


### 1.3 Configurar Variables de Entorno

Establecemos las rutas de Java y Spark en las variables de entorno del sistema.

In [3]:
import os

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'

os.environ['SPARK_HOME'] = '/content/spark-3.5.9-bin-hadoop3'

print('✓ Variables de entorno configuradas')

✓ Variables de entorno configuradas


### 1.4 Inicializar SparkSession y SparkContext



**SparkSession**: Punto de entrada principal para trabajar con Spark (incluye DataFrames y SQL).



**SparkContext**: Interfaz de bajo nivel para trabajar con RDDs.



A partir de Spark 2.0, SparkSession unifica SparkContext, SQLContext y HiveContext.

In [4]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('Capsula_RDD_Dataframe') \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext
print('✓ Sesión Spark creada exitosamente')
print(f'Spark Version: {spark.version}')
print(f'App Name: {spark.sparkContext.appName}')

✓ Sesión Spark creada exitosamente
Spark Version: 3.5.9
App Name: Capsula_RDD_Dataframe


# 2. REPASO: BIG DATA Y APACHE SPARK

## Conceptos Fundamentales de la Semana 1



Antes de profundizar en RDDs y DataFrames, recordemos los conceptos clave de la Semana 1.

## 2.1 ¿Qué es Big Data?



**Big Data** se refiere a conjuntos de datos que son tan grandes o complejos que las aplicaciones tradicionales de procesamiento de datos no son suficientes para manejarlos.



### Las 5 V's del Big Data:



1. **Volumen**: Cantidad masiva de datos (terabytes, petabytes, exabytes)

2. **Velocidad**: Rapidez con la que se generan y procesan los datos

3. **Variedad**: Diferentes tipos y formatos de datos (estructurados, semi-estructurados, no estructurados)

4. **Veracidad**: Calidad y confiabilidad de los datos

5. **Valor**: Capacidad de extraer información útil y generar valor de negocio

## 2.2 Apache Hadoop: MapReduce y HDFS



**Apache Hadoop** fue el primer framework ampliamente adoptado para procesamiento distribuido de Big Data.



### Componentes principales:



- **HDFS (Hadoop Distributed File System)**: Sistema de archivos distribuido que almacena datos en múltiples nodos

- **MapReduce**: Modelo de programación para procesamiento paralelo de grandes volúmenes de datos



### Limitaciones de MapReduce:

- Procesamiento basado en disco (lento)

- Requiere escribir resultados intermedios en disco

- Difícil de programar para algoritmos iterativos

- No optimizado para procesamiento en tiempo real

## 2.3 ¿Por qué Spark y no Hadoop?



**Apache Spark** surgió como una evolución de Hadoop, superando sus limitaciones principales.



### Ventajas de Spark sobre Hadoop:



1. **Procesamiento en memoria**: 100x más rápido que MapReduce para datos en memoria, 10x más rápido para datos en disco



2. **Evaluación perezosa (Lazy Evaluation)**: Optimiza el plan de ejecución antes de ejecutar



3. **APIs de alto nivel**: RDD, DataFrame, Dataset - más fáciles de usar que MapReduce



4. **Soporte para múltiples lenguajes**: Python, Scala, Java, R, SQL



5. **Ecosistema integrado**:

   - **Spark SQL**: Consultas SQL y DataFrames

   - **MLlib**: Machine Learning

   - **GraphX**: Procesamiento de grafos

   - **Structured Streaming**: Procesamiento de streams en tiempo real

## 2.4 Arquitectura de Apache Spark



Spark utiliza una arquitectura **maestro-trabajador** (master-worker):



### Componentes principales:



1. **Driver Program**:

   - Ejecuta la función `main()` de la aplicación

   - Crea el SparkContext

   - Convierte el código en un DAG (Directed Acyclic Graph)

   - Distribuye tareas a los executors



2. **Cluster Manager**:

   - Administra los recursos del cluster

   - Opciones: Standalone, YARN, Mesos, Kubernetes



3. **Executors**:

   - Procesos que ejecutan las tareas

   - Almacenan datos en caché cuando es necesario

   - Devuelven resultados al Driver



### Flujo de ejecución:

```

Driver Program → Cluster Manager → Executors → Tareas → Resultados → Driver

```

# 3. RDD: RESILIENT DISTRIBUTED DATASETS

## La Abstracción Fundamental de Apache Spark



**RDD (Resilient Distributed Dataset)** es la estructura de datos fundamental de Apache Spark. Es una colección inmutable y distribuida de objetos que puede ser procesada en paralelo.

## 3.1 ¿Qué es un RDD?



Un **RDD** es:



- **Resilient (Resiliente)**: Tolerante a fallos mediante linaje (lineage). Si una partición se pierde, Spark puede reconstruirla usando el linaje.



- **Distributed (Distribuido)**: Los datos están particionados automáticamente a través de los nodos del cluster.



- **Dataset (Conjunto de datos)**: Una colección de elementos que pueden ser procesados en paralelo.



### Representación visual:

```

RDD = [Partición 1] + [Partición 2] + [Partición 3] + ... + [Partición N]

       (Nodo 1)       (Nodo 2)       (Nodo 3)           (Nodo N)

```

## 3.2 Características Fundamentales de RDD



### 1. **Inmutabilidad**

Una vez creado un RDD, **no se puede modificar**. Cualquier transformación crea un **nuevo RDD**.



**¿Por qué inmutabilidad?**

- Facilita el procesamiento paralelo (no hay conflictos de escritura)

- Permite la recuperación ante fallos mediante linaje

- Simplifica el razonamiento sobre el código



### 2. **Distribución**

Los datos se dividen automáticamente en **particiones** que se distribuyen a través del cluster.



**Ventajas:**

- Procesamiento paralelo automático

- Escalabilidad horizontal

- Localidad de datos (procesamiento cerca de donde están los datos)



### 3. **Tolerancia a Fallos (Resiliencia)**

Spark mantiene el **linaje (lineage)** de cada RDD: el conjunto de transformaciones que se aplicaron para crearlo.



Si una partición se pierde (por fallo de un nodo), Spark puede **reconstruirla** aplicando las transformaciones del linaje.



**Linaje = DAG (Directed Acyclic Graph)**



### 4. **Evaluación Perezosa (Lazy Evaluation)**

Las **transformaciones** no se ejecutan inmediatamente. Spark construye un plan de ejecución (DAG) y solo lo ejecuta cuando se invoca una **acción**.



**Ventajas:**

- Optimización del plan de ejecución completo

- Reducción de pasos intermedios

- Mejor uso de recursos

# 4. TRANSFORMACIONES VS ACCIONES

## Los Dos Tipos de Operaciones en Spark



En Spark, todas las operaciones sobre RDDs se clasifican en dos categorías: **Transformaciones** y **Acciones**.

## 4.1 Transformaciones (Lazy)



Las **transformaciones** son operaciones que generan un **nuevo RDD** a partir de uno existente.



### Características:

- **No se ejecutan inmediatamente** (lazy evaluation)

- Spark solo registra la transformación en el DAG

- Retornan un nuevo RDD

- Permiten encadenar múltiples operaciones



### Ejemplos de transformaciones:

- `map(func)`: Aplica una función a cada elemento

- `filter(func)`: Filtra elementos según una condición

- `flatMap(func)`: Aplica función y aplana el resultado

- `reduceByKey(func)`: Agrega valores por clave

- `groupByKey()`: Agrupa valores por clave

- `sortByKey()`: Ordena pares clave-valor por clave

- `union(rdd)`: Une dos RDDs

- `distinct()`: Elimina duplicados



### Sintaxis general:

```python

nuevo_rdd = rdd_original.transformacion(parametros)

```

## 4.2 Acciones (Eager)



Las **acciones** son operaciones que **retornan un resultado** al Driver Program o escriben datos en un sistema de almacenamiento.



### Características:

- **Se ejecutan inmediatamente** (eager evaluation)

- Desencadenan la ejecución de todas las transformaciones pendientes en el DAG

- Retornan un valor (no un RDD)

- Pueden escribir datos en disco



### Ejemplos de acciones:

- `count()`: Cuenta el número de elementos

- `collect()`: Devuelve todos los elementos como array

- `reduce(func)`: Agrega elementos usando una función

- `take(n)`: Devuelve los primeros n elementos

- `first()`: Devuelve el primer elemento

- `saveAsTextFile(path)`: Guarda el RDD en archivos de texto

- `foreach(func)`: Aplica función a cada elemento (sin retornar resultado)



### Sintaxis general:

```python

resultado = rdd.accion()

```

## 4.3 Diferencia Clave: Lazy vs Eager



| Aspecto | Transformaciones (Lazy) | Acciones (Eager) |

|---------|------------------------|------------------|

| **Ejecución** | Diferida (lazy) | Inmediata (eager) |

| **Retorno** | Nuevo RDD | Valor o escritura |

| **Efecto** | Registra operación en DAG | Ejecuta DAG completo |

| **Ejemplos** | map, filter, reduceByKey | count, collect, save |



### Ejemplo conceptual:

```python

# TRANSFORMACIONES (no se ejecutan aún)

rdd2 = rdd1.map(lambda x: x * 2)        # Lazy

rdd3 = rdd2.filter(lambda x: x > 10)    # Lazy

rdd4 = rdd3.distinct()                  # Lazy



# ACCIÓN (ejecuta todo el DAG)

resultado = rdd4.count()                # Eager - ¡AQUÍ SE EJECUTA TODO!

```

# 5. CREACIÓN DE RDDs

## Formas de Crear RDDs en Spark



Existen tres formas principales de crear RDDs en Spark.

## 5.1 Desde una Colección en Python



Podemos crear un RDD a partir de una lista, tupla o cualquier colección iterable de Python usando `parallelize()`.

In [11]:
# Crear RDD desde una lista de números

numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

rdd_numeros = sc.parallelize(numeros)



print('RDD creado desde lista:')

print(f'Tipo: {type(rdd_numeros)}')

print(f'Número de particiones: {rdd_numeros.getNumPartitions()}')

print(f'Primeros 5 elementos: {rdd_numeros.take(5)}')

RDD creado desde lista:
Tipo: <class 'pyspark.rdd.RDD'>
Número de particiones: 2
Primeros 5 elementos: [1, 2, 3, 4, 5]


In [12]:
# Crear RDD desde una lista de tuplas (pares clave-valor)

pares = [('manzana', 5), ('naranja', 3), ('pera', 8), ('manzana', 2)]

rdd_pares = sc.parallelize(pares)



print('RDD de pares clave-valor:')

print(f'Elementos: {rdd_pares.collect()}')

RDD de pares clave-valor:
Elementos: [('manzana', 5), ('naranja', 3), ('pera', 8), ('manzana', 2)]


## 5.2 Desde un Archivo de Texto



Podemos crear un RDD leyendo archivos de texto usando `textFile()`. Cada línea del archivo se convierte en un elemento del RDD.

In [13]:
# Crear un archivo de texto de ejemplo

texto_ejemplo = """Apache Spark es un motor de procesamiento distribuido

Spark es más rápido que Hadoop MapReduce

RDD es la abstracción fundamental de Spark

Spark soporta Python Scala Java y R

Big Data requiere procesamiento distribuido"""



# Guardar en archivo

with open('/content/ejemplo.txt', 'w') as f:

    f.write(texto_ejemplo)



# Crear RDD desde archivo

rdd_texto = sc.textFile('/content/ejemplo.txt')



print('RDD creado desde archivo de texto:')

print(f'Número de líneas: {rdd_texto.count()}')

print(f'Primera línea: {rdd_texto.first()}')

print(f'\nTodas las líneas:')

for linea in rdd_texto.collect():

    print(f'  - {linea}')

RDD creado desde archivo de texto:
Número de líneas: 9
Primera línea: Apache Spark es un motor de procesamiento distribuido

Todas las líneas:
  - Apache Spark es un motor de procesamiento distribuido
  - 
  - Spark es más rápido que Hadoop MapReduce
  - 
  - RDD es la abstracción fundamental de Spark
  - 
  - Spark soporta Python Scala Java y R
  - 
  - Big Data requiere procesamiento distribuido


## 5.3 Desde Otro RDD (mediante transformaciones)



La forma más común de crear RDDs es aplicando transformaciones a RDDs existentes.

In [14]:
# Crear nuevo RDD mediante transformación

rdd_original = sc.parallelize([1, 2, 3, 4, 5])

rdd_transformado = rdd_original.map(lambda x: x * 2)



print('RDD original:', rdd_original.collect())

print('RDD transformado:', rdd_transformado.collect())

RDD original: [1, 2, 3, 4, 5]
RDD transformado: [2, 4, 6, 8, 10]


# 6. TRANSFORMACIONES COMUNES

## Operaciones Lazy sobre RDDs



Las transformaciones son el corazón del procesamiento con Spark. Veamos las más utilizadas con ejemplos prácticos.

## 6.1 map(func)



**Descripción**: Aplica una función a cada elemento del RDD y retorna un nuevo RDD con los resultados.



**Sintaxis**: `rdd.map(lambda x: expresion)`



**Característica**: 1 elemento de entrada → 1 elemento de salida

In [15]:
# Ejemplo 1: Multiplicar cada número por 2

numeros = sc.parallelize([1, 2, 3, 4, 5])

numeros_dobles = numeros.map(lambda x: x * 2)



print('Original:', numeros.collect())

print('Dobles:', numeros_dobles.collect())

Original: [1, 2, 3, 4, 5]
Dobles: [2, 4, 6, 8, 10]


In [16]:
# Ejemplo 2: Convertir texto a mayúsculas

palabras = sc.parallelize(['hola', 'mundo', 'spark', 'python'])

palabras_mayusculas = palabras.map(lambda x: x.upper())



print('Original:', palabras.collect())

print('Mayúsculas:', palabras_mayusculas.collect())

Original: ['hola', 'mundo', 'spark', 'python']
Mayúsculas: ['HOLA', 'MUNDO', 'SPARK', 'PYTHON']


In [17]:
# Ejemplo 3: Extraer longitud de cada palabra

palabras = sc.parallelize(['Apache', 'Spark', 'RDD', 'DataFrame'])

longitudes = palabras.map(lambda x: (x, len(x)))



print('Palabras con longitud:')

for palabra, longitud in longitudes.collect():

    print(f'  {palabra}: {longitud} caracteres')

Palabras con longitud:
  Apache: 6 caracteres
  Spark: 5 caracteres
  RDD: 3 caracteres
  DataFrame: 9 caracteres


## 6.2 filter(func)



**Descripción**: Filtra elementos del RDD que cumplen una condición (retorna True).



**Sintaxis**: `rdd.filter(lambda x: condicion)`



**Característica**: Retorna solo los elementos que cumplen la condición

In [18]:
# Ejemplo 1: Filtrar números pares

numeros = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

pares = numeros.filter(lambda x: x % 2 == 0)



print('Números:', numeros.collect())

print('Pares:', pares.collect())

Números: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Pares: [2, 4, 6, 8, 10]


In [19]:
# Ejemplo 2: Filtrar palabras largas (más de 5 caracteres)

palabras = sc.parallelize(['Python', 'Spark', 'RDD', 'DataFrame', 'Big', 'Data'])

palabras_largas = palabras.filter(lambda x: len(x) > 5)



print('Todas las palabras:', palabras.collect())

print('Palabras largas (>5 caracteres):', palabras_largas.collect())

Todas las palabras: ['Python', 'Spark', 'RDD', 'DataFrame', 'Big', 'Data']
Palabras largas (>5 caracteres): ['Python', 'DataFrame']


In [20]:
# Ejemplo 3: Filtrar líneas que contienen una palabra específica

lineas = sc.parallelize([

    'Apache Spark es rápido',

    'Python es un lenguaje popular',

    'Spark soporta Python',

    'Big Data requiere Spark'

])

lineas_spark = lineas.filter(lambda x: 'Spark' in x)



print('Líneas que contienen "Spark":')

for linea in lineas_spark.collect():

    print(f'  - {linea}')

Líneas que contienen "Spark":
  - Apache Spark es rápido
  - Spark soporta Python
  - Big Data requiere Spark


## 6.3 flatMap(func)



**Descripción**: Similar a `map()`, pero aplana el resultado. Cada elemento puede generar 0, 1 o múltiples elementos de salida.



**Sintaxis**: `rdd.flatMap(lambda x: lista_o_iterador)`



**Característica**: 1 elemento de entrada → 0, 1 o N elementos de salida (aplanados)



**Diferencia con map()**:

- `map()`: `[1, 2, 3]` → `[[a,b], [c,d], [e,f]]` (lista de listas)

- `flatMap()`: `[1, 2, 3]` → `[a, b, c, d, e, f]` (lista plana)

In [21]:
# Ejemplo 1: Dividir líneas en palabras

lineas = sc.parallelize([

    'Apache Spark es rápido',

    'Python es popular',

    'Big Data con Spark'

])



# Con map() (lista de listas)

palabras_map = lineas.map(lambda x: x.split())

print('Con map() (lista de listas):', palabras_map.collect())



# Con flatMap() (lista plana)

palabras_flatmap = lineas.flatMap(lambda x: x.split())

print('Con flatMap() (lista plana):', palabras_flatmap.collect())

Con map() (lista de listas): [['Apache', 'Spark', 'es', 'rápido'], ['Python', 'es', 'popular'], ['Big', 'Data', 'con', 'Spark']]
Con flatMap() (lista plana): ['Apache', 'Spark', 'es', 'rápido', 'Python', 'es', 'popular', 'Big', 'Data', 'con', 'Spark']


In [22]:
# Ejemplo 2: Generar múltiples elementos por cada entrada

numeros = sc.parallelize([1, 2, 3])



# Generar rango de números para cada elemento

rangos = numeros.flatMap(lambda x: range(1, x + 1))



print('Números originales:', numeros.collect())

print('Rangos aplanados:', rangos.collect())

# 1 → [1]

# 2 → [1, 2]

# 3 → [1, 2, 3]

# Resultado aplanado: [1, 1, 2, 1, 2, 3]

Números originales: [1, 2, 3]
Rangos aplanados: [1, 1, 2, 1, 2, 3]


## 6.4 reduceByKey(func)



**Descripción**: Agrega valores de pares clave-valor que tienen la misma clave, usando una función de reducción.



**Sintaxis**: `rdd_pares.reduceByKey(lambda a, b: operacion)`



**Requisito**: El RDD debe contener tuplas (clave, valor)



**Característica**: Combina valores por clave de forma distribuida (más eficiente que groupByKey)

In [23]:
# Ejemplo 1: Sumar valores por clave

pares = sc.parallelize([

    ('manzana', 5),

    ('naranja', 3),

    ('manzana', 2),

    ('pera', 8),

    ('naranja', 4),

    ('manzana', 1)

])



totales = pares.reduceByKey(lambda a, b: a + b)



print('Pares originales:')

for clave, valor in pares.collect():

    print(f'  {clave}: {valor}')



print('\nTotales por fruta:')

for fruta, total in totales.collect():

    print(f'  {fruta}: {total}')

Pares originales:
  manzana: 5
  naranja: 3
  manzana: 2
  pera: 8
  naranja: 4
  manzana: 1

Totales por fruta:
  naranja: 7
  pera: 8
  manzana: 8


In [24]:
# Ejemplo 2: Encontrar el máximo por clave

temperaturas = sc.parallelize([

    ('Santiago', 25),

    ('Valparaíso', 22),

    ('Santiago', 28),

    ('Concepción', 20),

    ('Valparaíso', 24),

    ('Santiago', 26)

])



temp_maximas = temperaturas.reduceByKey(lambda a, b: max(a, b))



print('Temperaturas máximas por ciudad:')

for ciudad, temp in temp_maximas.collect():

    print(f'  {ciudad}: {temp}°C')

Temperaturas máximas por ciudad:
  Santiago: 28°C
  Valparaíso: 24°C
  Concepción: 20°C


## 6.5 groupByKey()



**Descripción**: Agrupa valores de pares clave-valor que tienen la misma clave.



**Sintaxis**: `rdd_pares.groupByKey()`



**Diferencia con reduceByKey()**:

- `groupByKey()`: Agrupa todos los valores en una lista (menos eficiente)

- `reduceByKey()`: Reduce valores de forma distribuida (más eficiente)



**Recomendación**: Usar `reduceByKey()` cuando sea posible

In [25]:
# Ejemplo: Agrupar valores por clave

pares = sc.parallelize([

    ('A', 1),

    ('B', 2),

    ('A', 3),

    ('B', 4),

    ('A', 5)

])



grupos = pares.groupByKey()



print('Valores agrupados por clave:')

for clave, valores in grupos.collect():

    print(f'  {clave}: {list(valores)}')

Valores agrupados por clave:
  A: [1, 3, 5]
  B: [2, 4]


## 6.6 sortByKey()



**Descripción**: Ordena pares clave-valor por clave.



**Sintaxis**: `rdd_pares.sortByKey(ascending=True)`



**Parámetros**:

- `ascending=True`: Orden ascendente (por defecto)

- `ascending=False`: Orden descendente

In [26]:
# Ejemplo: Ordenar por clave

pares = sc.parallelize([

    ('zebra', 10),

    ('apple', 5),

    ('mango', 8),

    ('banana', 3)

])



# Orden ascendente

ordenado_asc = pares.sortByKey(ascending=True)

print('Orden ascendente (A-Z):', ordenado_asc.collect())



# Orden descendente

ordenado_desc = pares.sortByKey(ascending=False)

print('Orden descendente (Z-A):', ordenado_desc.collect())

Orden ascendente (A-Z): [('apple', 5), ('banana', 3), ('mango', 8), ('zebra', 10)]
Orden descendente (Z-A): [('zebra', 10), ('mango', 8), ('banana', 3), ('apple', 5)]


## 6.7 distinct()



**Descripción**: Elimina elementos duplicados del RDD.



**Sintaxis**: `rdd.distinct()`

In [27]:
# Ejemplo: Eliminar duplicados

numeros = sc.parallelize([1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5])



print('Con duplicados:', numeros.collect())

print('Sin duplicados:', numeros.distinct().collect())

print(f'\nTotal original: {numeros.count()} elementos')

print(f'Total sin duplicados: {numeros.distinct().count()} elementos')

Con duplicados: [1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5]
Sin duplicados: [2, 4, 1, 3, 5]

Total original: 11 elementos
Total sin duplicados: 5 elementos


## 6.8 union()



**Descripción**: Une dos RDDs (concatenación, no elimina duplicados).



**Sintaxis**: `rdd1.union(rdd2)`

In [28]:
# Ejemplo: Unir dos RDDs

rdd1 = sc.parallelize([1, 2, 3, 4])

rdd2 = sc.parallelize([3, 4, 5, 6])



rdd_union = rdd1.union(rdd2)



print('RDD 1:', rdd1.collect())

print('RDD 2:', rdd2.collect())

print('Unión (con duplicados):', rdd_union.collect())

print('Unión sin duplicados:', rdd_union.distinct().collect())

RDD 1: [1, 2, 3, 4]
RDD 2: [3, 4, 5, 6]
Unión (con duplicados): [1, 2, 3, 4, 3, 4, 5, 6]
Unión sin duplicados: [4, 1, 5, 2, 6, 3]


# 7. ACCIONES COMUNES

## Operaciones Eager que Ejecutan el DAG



Las acciones desencadenan la ejecución de todas las transformaciones pendientes y retornan un resultado.

## 7.1 count()



**Descripción**: Cuenta el número de elementos en el RDD.



**Sintaxis**: `rdd.count()`



**Retorna**: Entero (número de elementos)

In [29]:
# Ejemplo: Contar elementos

numeros = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

pares = numeros.filter(lambda x: x % 2 == 0)



print(f'Total de números: {numeros.count()}')

print(f'Total de pares: {pares.count()}')

Total de números: 10
Total de pares: 5


## 7.2 collect()



**Descripción**: Devuelve todos los elementos del RDD como una lista de Python.



**Sintaxis**: `rdd.collect()`



**⚠️ ADVERTENCIA**: Solo usar con RDDs pequeños. Si el RDD es muy grande, puede agotar la memoria del Driver.

In [30]:
# Ejemplo: Recolectar todos los elementos

numeros = sc.parallelize([1, 2, 3, 4, 5])

cuadrados = numeros.map(lambda x: x ** 2)



resultado = cuadrados.collect()

print('Tipo:', type(resultado))

print('Cuadrados:', resultado)

Tipo: <class 'list'>
Cuadrados: [1, 4, 9, 16, 25]


## 7.3 take(n)



**Descripción**: Devuelve los primeros n elementos del RDD.



**Sintaxis**: `rdd.take(n)`



**Ventaja**: Más seguro que `collect()` para RDDs grandes

In [31]:
# Ejemplo: Tomar los primeros elementos

numeros = sc.parallelize(range(1, 101))  # 1 a 100



print('Primeros 5 números:', numeros.take(5))

print('Primeros 10 números:', numeros.take(10))

Primeros 5 números: [1, 2, 3, 4, 5]
Primeros 10 números: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 7.4 first()



**Descripción**: Devuelve el primer elemento del RDD.



**Sintaxis**: `rdd.first()`



**Equivalente a**: `rdd.take(1)[0]`

In [32]:
# Ejemplo: Obtener el primer elemento

palabras = sc.parallelize(['Apache', 'Spark', 'Python', 'RDD'])



print('Primera palabra:', palabras.first())

Primera palabra: Apache


## 7.5 reduce(func)



**Descripción**: Agrega todos los elementos del RDD usando una función binaria.



**Sintaxis**: `rdd.reduce(lambda a, b: operacion)`



**Requisito**: La función debe ser asociativa y conmutativa

In [33]:
# Ejemplo 1: Sumar todos los números

numeros = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

suma = numeros.reduce(lambda a, b: a + b)



print(f'Suma de 1 a 10: {suma}')

Suma de 1 a 10: 55


In [34]:
# Ejemplo 2: Encontrar el máximo

numeros = sc.parallelize([45, 23, 67, 12, 89, 34, 56])

maximo = numeros.reduce(lambda a, b: max(a, b))



print(f'Número máximo: {maximo}')

Número máximo: 89


## 7.6 saveAsTextFile(path)



**Descripción**: Guarda el RDD en archivos de texto en el sistema de archivos.



**Sintaxis**: `rdd.saveAsTextFile(path)`



**Nota**: Crea un directorio con múltiples archivos (uno por partición)

In [35]:
# Ejemplo: Guardar RDD en archivo

palabras = sc.parallelize(['Apache', 'Spark', 'Python', 'RDD', 'DataFrame'])



# Guardar en directorio

output_path = '/content/output_palabras'



# Eliminar directorio si existe

import shutil

import os

if os.path.exists(output_path):

    shutil.rmtree(output_path)



palabras.saveAsTextFile(output_path)



print(f'✓ RDD guardado en: {output_path}')

print('\nArchivos generados:')

!ls -lh {output_path}



print('\nContenido del primer archivo:')

!cat {output_path}/part-00000

✓ RDD guardado en: /content/output_palabras

Archivos generados:
total 8.0K
-rw-r--r-- 1 root root 13 Jul 20 13:51 part-00000
-rw-r--r-- 1 root root 21 Jul 20 13:51 part-00001
-rw-r--r-- 1 root root  0 Jul 20 13:51 _SUCCESS

Contenido del primer archivo:
Apache
Spark


# 8. EVALUACIÓN PEREZOSA (LAZY EVALUATION)

## Demostración Práctica del DAG



La evaluación perezosa es una de las características más importantes de Spark. Veamos cómo funciona en la práctica.

## 8.1 ¿Qué es la Evaluación Perezosa?



**Evaluación Perezosa** significa que las **transformaciones no se ejecutan inmediatamente**. En su lugar, Spark:



1. **Registra** la transformación en un plan de ejecución (DAG - Directed Acyclic Graph)

2. **Optimiza** el plan completo

3. **Ejecuta** todo cuando se invoca una **acción**



### Ventajas:

- **Optimización**: Spark puede optimizar el plan completo antes de ejecutar

- **Eficiencia**: Elimina pasos intermedios innecesarios

- **Recursos**: Mejor uso de memoria y CPU



### Ejemplo conceptual:

```python

rdd1 = sc.parallelize([1, 2, 3, 4, 5])     # Crea RDD

rdd2 = rdd1.map(lambda x: x * 2)           # LAZY - no se ejecuta

rdd3 = rdd2.filter(lambda x: x > 5)        # LAZY - no se ejecuta

rdd4 = rdd3.map(lambda x: x + 1)           # LAZY - no se ejecuta



resultado = rdd4.collect()                 # EAGER - ¡EJECUTA TODO!

```

## 8.2 Demostración Práctica



Vamos a demostrar que las transformaciones no se ejecutan hasta que se invoca una acción.

In [36]:
import time



print('=== DEMOSTRACIÓN DE EVALUACIÓN PEREZOSA ===')

print()



# Crear RDD

print('1. Creando RDD...')

start = time.time()

rdd1 = sc.parallelize(range(1, 1000001))  # 1 millón de números

print(f'   Tiempo: {time.time() - start:.4f} segundos')

print('   ✓ RDD creado (pero no se ha procesado ningún dato)')

print()



# Aplicar transformaciones

print('2. Aplicando transformaciones...')

start = time.time()

rdd2 = rdd1.map(lambda x: x * 2)

print(f'   map() - Tiempo: {time.time() - start:.4f} segundos')



start = time.time()

rdd3 = rdd2.filter(lambda x: x > 1000000)

print(f'   filter() - Tiempo: {time.time() - start:.4f} segundos')



start = time.time()

rdd4 = rdd3.map(lambda x: x + 1)

print(f'   map() - Tiempo: {time.time() - start:.4f} segundos')



print('   ✓ Transformaciones registradas (pero NO ejecutadas)')

print('   ✓ Spark solo ha construido el DAG')

print()



# Invocar acción

print('3. Invocando acción count()...')

start = time.time()

resultado = rdd4.count()

tiempo_ejecucion = time.time() - start

print(f'   Tiempo: {tiempo_ejecucion:.4f} segundos')

print(f'   ✓ ¡AQUÍ SE EJECUTÓ TODO EL DAG!')

print(f'   Resultado: {resultado} elementos')

print()



print('CONCLUSIÓN:')

print('- Las transformaciones (map, filter) fueron instantáneas (lazy)')

print(f'- La acción (count) tomó {tiempo_ejecucion:.4f}s porque ejecutó todo el DAG')

=== DEMOSTRACIÓN DE EVALUACIÓN PEREZOSA ===

1. Creando RDD...
   Tiempo: 0.0072 segundos
   ✓ RDD creado (pero no se ha procesado ningún dato)

2. Aplicando transformaciones...
   map() - Tiempo: 0.0002 segundos
   filter() - Tiempo: 0.0002 segundos
   map() - Tiempo: 0.0001 segundos
   ✓ Transformaciones registradas (pero NO ejecutadas)
   ✓ Spark solo ha construido el DAG

3. Invocando acción count()...
   Tiempo: 0.7956 segundos
   ✓ ¡AQUÍ SE EJECUTÓ TODO EL DAG!
   Resultado: 500000 elementos

CONCLUSIÓN:
- Las transformaciones (map, filter) fueron instantáneas (lazy)
- La acción (count) tomó 0.7956s porque ejecutó todo el DAG


## 8.3 Persistencia (Caché)



Cuando un RDD se usa múltiples veces, podemos **persistirlo en memoria** usando `cache()` o `persist()` para evitar recalcularlo.



**Sintaxis**:

- `rdd.cache()`: Almacena en memoria

- `rdd.persist(StorageLevel)`: Almacena con nivel de almacenamiento específico



**Cuándo usar caché:**

- Cuando un RDD se usa en múltiples acciones

- Cuando el cálculo del RDD es costoso

- Cuando hay suficiente memoria disponible

In [37]:
import time



print('=== DEMOSTRACIÓN DE CACHÉ ===')

print()



# Crear RDD con operación costosa

rdd = sc.parallelize(range(1, 10000001))  # 10 millones

rdd_procesado = rdd.map(lambda x: x * 2).filter(lambda x: x > 5000000)



# Primera ejecución SIN caché

print('1. Primera ejecución SIN caché:')

start = time.time()

count1 = rdd_procesado.count()

tiempo1 = time.time() - start

print(f'   Tiempo: {tiempo1:.4f} segundos')

print(f'   Resultado: {count1} elementos')

print()



# Segunda ejecución SIN caché (recalcula todo)

print('2. Segunda ejecución SIN caché:')

start = time.time()

sum1 = rdd_procesado.reduce(lambda a, b: a + b)

tiempo2 = time.time() - start

print(f'   Tiempo: {tiempo2:.4f} segundos')

print(f'   (Recalculó todo el RDD)')

print()



# Ahora CON caché

print('3. Aplicando caché...')

rdd_procesado.cache()

print('   ✓ Caché activado')

print()



# Primera ejecución con caché (carga en memoria)

print('4. Primera ejecución CON caché (carga en memoria):')

start = time.time()

count2 = rdd_procesado.count()

tiempo3 = time.time() - start

print(f'   Tiempo: {tiempo3:.4f} segundos')

print()



# Segunda ejecución con caché (usa memoria)

print('5. Segunda ejecución CON caché (usa memoria):')

start = time.time()

sum2 = rdd_procesado.reduce(lambda a, b: a + b)

tiempo4 = time.time() - start

print(f'   Tiempo: {tiempo4:.4f} segundos')

print(f'   ¡Mucho más rápido! (usó datos en caché)')

print()



# Limpiar caché

rdd_procesado.unpersist()

print('✓ Caché liberado')

=== DEMOSTRACIÓN DE CACHÉ ===

1. Primera ejecución SIN caché:
   Tiempo: 5.2190 segundos
   Resultado: 7500000 elementos

2. Segunda ejecución SIN caché:
   Tiempo: 9.2681 segundos
   (Recalculó todo el RDD)

3. Aplicando caché...
   ✓ Caché activado

4. Primera ejecución CON caché (carga en memoria):
   Tiempo: 6.4438 segundos

5. Segunda ejecución CON caché (usa memoria):
   Tiempo: 2.7632 segundos
   ¡Mucho más rápido! (usó datos en caché)

✓ Caché liberado


# 9. EJEMPLO COMPLETO: WORDCOUNT

## El "Hello World" de Big Data



**WordCount** es el ejemplo clásico de procesamiento distribuido. Cuenta la frecuencia de cada palabra en un texto.

## 9.1 Problema



Dado un texto, queremos:

1. Contar cuántas veces aparece cada palabra

2. Ordenar las palabras por frecuencia (de mayor a menor)

3. Mostrar las 10 palabras más frecuentes



### Pasos del algoritmo:

1. Leer el texto

2. Dividir en palabras

3. Convertir a minúsculas y limpiar

4. Crear pares (palabra, 1)

5. Sumar por palabra usando `reduceByKey()`

6. Ordenar por frecuencia

7. Mostrar resultados

## 9.2 Implementación Paso a Paso

In [38]:
# Paso 1: Crear texto de ejemplo

texto = """Apache Spark es un motor de procesamiento de datos distribuido y de código abierto

Spark es más rápido que Hadoop MapReduce porque procesa datos en memoria

RDD es la abstracción fundamental de Spark para representar datos distribuidos

Spark soporta múltiples lenguajes como Python Scala Java y R

Big Data requiere herramientas de procesamiento distribuido como Spark

Spark SQL permite consultar datos usando SQL

MLlib es la librería de machine learning de Spark

GraphX es el componente de Spark para procesamiento de grafos

Structured Streaming permite procesamiento de streams en tiempo real con Spark

Spark es ampliamente usado en la industria para análisis de Big Data"""



# Guardar en archivo

with open('/content/texto_spark.txt', 'w', encoding='utf-8') as f:

    f.write(texto)



print('✓ Texto de ejemplo creado')

print(f'Longitud del texto: {len(texto)} caracteres')

✓ Texto de ejemplo creado
Longitud del texto: 680 caracteres


In [39]:
# Paso 2: Leer el archivo como RDD

lineas = sc.textFile('/content/texto_spark.txt')



print(f'Número de líneas: {lineas.count()}')

print('\nPrimeras 3 líneas:')

for i, linea in enumerate(lineas.take(3), 1):

    print(f'{i}. {linea}')

Número de líneas: 19

Primeras 3 líneas:
1. Apache Spark es un motor de procesamiento de datos distribuido y de código abierto
2. 
3. Spark es más rápido que Hadoop MapReduce porque procesa datos en memoria


In [40]:
# Paso 3: Dividir líneas en palabras (flatMap)

palabras = lineas.flatMap(lambda linea: linea.split())



print(f'Total de palabras (con repeticiones): {palabras.count()}')

print(f'Primeras 10 palabras: {palabras.take(10)}')

Total de palabras (con repeticiones): 105
Primeras 10 palabras: ['Apache', 'Spark', 'es', 'un', 'motor', 'de', 'procesamiento', 'de', 'datos', 'distribuido']


# 10. INTRODUCCIÓN A DATAFRAMES

## Abstracción de Alto Nivel sobre RDDs



Los **DataFrames** son una evolución de los RDDs que proporcionan una API más intuitiva y mejor rendimiento.

## 10.1 ¿Qué es un DataFrame?



Un **DataFrame** es:

- Una colección distribuida de datos organizados en **columnas nombradas**

- Similar a una tabla de base de datos relacional o un DataFrame de Pandas

- Construido sobre RDDs pero con optimizaciones automáticas



### Ventajas sobre RDDs:



1. **API más intuitiva**: Similar a SQL y Pandas

2. **Optimización automática**: Catalyst Optimizer optimiza las consultas

3. **Mejor rendimiento**: Tungsten execution engine

4. **Esquema estructurado**: Columnas con tipos de datos

5. **Inferencia de esquema**: Detecta tipos automáticamente

6. **Múltiples formatos**: CSV, JSON, Parquet, JDBC, etc.



### Cuándo usar DataFrames vs RDDs:



| Usar DataFrames | Usar RDDs |

|-----------------|----------|

| Datos estructurados/semi-estructurados | Datos no estructurados |

| Operaciones tipo SQL | Control fino del procesamiento |

| Mejor rendimiento automático | Necesitas control total |

| API de alto nivel | Operaciones de bajo nivel |

## 10.2 Crear DataFrames

In [41]:
# Método 1: Desde una lista de tuplas

datos = [

    ('Juan', 25, 'Ingeniero'),

    ('María', 30, 'Doctora'),

    ('Pedro', 28, 'Profesor'),

    ('Ana', 35, 'Arquitecta')

]



columnas = ['nombre', 'edad', 'profesion']



df = spark.createDataFrame(datos, columnas)



print('DataFrame creado desde lista:')

df.show()

DataFrame creado desde lista:
+------+----+----------+
|nombre|edad| profesion|
+------+----+----------+
|  Juan|  25| Ingeniero|
| María|  30|   Doctora|
| Pedro|  28|  Profesor|
|   Ana|  35|Arquitecta|
+------+----+----------+



In [42]:
# Método 2: Desde un RDD

rdd = sc.parallelize([

    (1, 'Producto A', 100),

    (2, 'Producto B', 150),

    (3, 'Producto C', 200)

])



df_productos = spark.createDataFrame(rdd, ['id', 'nombre', 'precio'])



print('DataFrame creado desde RDD:')

df_productos.show()

DataFrame creado desde RDD:
+---+----------+------+
| id|    nombre|precio|
+---+----------+------+
|  1|Producto A|   100|
|  2|Producto B|   150|
|  3|Producto C|   200|
+---+----------+------+



In [43]:
# Método 3: Desde un archivo CSV

# Crear archivo CSV de ejemplo

csv_data = """nombre,edad,ciudad,salario

Juan,25,Santiago,1500000

María,30,Valparaíso,2000000

Pedro,28,Concepción,1800000

Ana,35,Santiago,2500000

Luis,22,Viña del Mar,1400000"""



with open('/content/empleados.csv', 'w') as f:

    f.write(csv_data)



# Leer CSV

df_empleados = spark.read.csv('/content/empleados.csv', header=True, inferSchema=True)



print('DataFrame creado desde CSV:')

df_empleados.show()

DataFrame creado desde CSV:
+------+----+------------+-------+
|nombre|edad|      ciudad|salario|
+------+----+------------+-------+
|  Juan|  25|    Santiago|1500000|
| María|  30|  Valparaíso|2000000|
| Pedro|  28|  Concepción|1800000|
|   Ana|  35|    Santiago|2500000|
|  Luis|  22|Viña del Mar|1400000|
+------+----+------------+-------+



In [44]:
# Método 4: Desde un archivo JSON

import json



# Crear archivo JSON de ejemplo

json_data = [

    {'producto': 'Laptop', 'precio': 800000, 'stock': 15},

    {'producto': 'Mouse', 'precio': 15000, 'stock': 50},

    {'producto': 'Teclado', 'precio': 35000, 'stock': 30},

    {'producto': 'Monitor', 'precio': 250000, 'stock': 20}

]



with open('/content/productos.json', 'w') as f:

    for item in json_data:

        f.write(json.dumps(item) + '\n')



# Leer JSON

df_productos_json = spark.read.json('/content/productos.json')



print('DataFrame creado desde JSON:')

df_productos_json.show()

DataFrame creado desde JSON:
+------+--------+-----+
|precio|producto|stock|
+------+--------+-----+
|800000|  Laptop|   15|
| 15000|   Mouse|   50|
| 35000| Teclado|   30|
|250000| Monitor|   20|
+------+--------+-----+



## 10.3 Operaciones Básicas con DataFrames

In [45]:
# Ver esquema del DataFrame

print('=== ESQUEMA DEL DATAFRAME ===')

df_empleados.printSchema()



# Información básica

print('\n=== INFORMACIÓN BÁSICA ===')

print(f'Número de filas: {df_empleados.count()}')

print(f'Número de columnas: {len(df_empleados.columns)}')

print(f'Columnas: {df_empleados.columns}')

=== ESQUEMA DEL DATAFRAME ===
root
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- salario: integer (nullable = true)


=== INFORMACIÓN BÁSICA ===
Número de filas: 5
Número de columnas: 4
Columnas: ['nombre', 'edad', 'ciudad', 'salario']


In [46]:
# Seleccionar columnas

print('=== SELECCIONAR COLUMNAS ===')

df_empleados.select('nombre', 'edad').show()



# Filtrar filas

print('\n=== FILTRAR FILAS (edad > 25) ===')

df_empleados.filter(df_empleados['edad'] > 25).show()



# Filtrar con SQL-like syntax

print('\n=== FILTRAR CON WHERE (ciudad = Santiago) ===')

df_empleados.where(df_empleados['ciudad'] == 'Santiago').show()

=== SELECCIONAR COLUMNAS ===
+------+----+
|nombre|edad|
+------+----+
|  Juan|  25|
| María|  30|
| Pedro|  28|
|   Ana|  35|
|  Luis|  22|
+------+----+


=== FILTRAR FILAS (edad > 25) ===
+------+----+----------+-------+
|nombre|edad|    ciudad|salario|
+------+----+----------+-------+
| María|  30|Valparaíso|2000000|
| Pedro|  28|Concepción|1800000|
|   Ana|  35|  Santiago|2500000|
+------+----+----------+-------+


=== FILTRAR CON WHERE (ciudad = Santiago) ===
+------+----+--------+-------+
|nombre|edad|  ciudad|salario|
+------+----+--------+-------+
|  Juan|  25|Santiago|1500000|
|   Ana|  35|Santiago|2500000|
+------+----+--------+-------+



In [47]:
# Agregar nueva columna

from pyspark.sql.functions import col



print('=== AGREGAR COLUMNA CALCULADA ===')

df_con_bonus = df_empleados.withColumn('salario_con_bonus', col('salario') * 1.1)

df_con_bonus.show()

=== AGREGAR COLUMNA CALCULADA ===
+------+----+------------+-------+------------------+
|nombre|edad|      ciudad|salario| salario_con_bonus|
+------+----+------------+-------+------------------+
|  Juan|  25|    Santiago|1500000|1650000.0000000002|
| María|  30|  Valparaíso|2000000|         2200000.0|
| Pedro|  28|  Concepción|1800000|1980000.0000000002|
|   Ana|  35|    Santiago|2500000|         2750000.0|
|  Luis|  22|Viña del Mar|1400000|1540000.0000000002|
+------+----+------------+-------+------------------+



In [48]:
# Agrupar y agregar

print('=== AGRUPAR POR CIUDAD Y CALCULAR PROMEDIO ===')

df_empleados.groupBy('ciudad').avg('salario').show()



print('\n=== ESTADÍSTICAS POR CIUDAD ===')

from pyspark.sql.functions import avg, max, min, count



df_empleados.groupBy('ciudad').agg(

    count('nombre').alias('cantidad'),

    avg('salario').alias('salario_promedio'),

    max('salario').alias('salario_maximo'),

    min('edad').alias('edad_minima')

).show()

=== AGRUPAR POR CIUDAD Y CALCULAR PROMEDIO ===
+------------+------------+
|      ciudad|avg(salario)|
+------------+------------+
|  Concepción|   1800000.0|
|Viña del Mar|   1400000.0|
|  Valparaíso|   2000000.0|
|    Santiago|   2000000.0|
+------------+------------+


=== ESTADÍSTICAS POR CIUDAD ===
+------------+--------+----------------+--------------+-----------+
|      ciudad|cantidad|salario_promedio|salario_maximo|edad_minima|
+------------+--------+----------------+--------------+-----------+
|  Concepción|       1|       1800000.0|       1800000|         28|
|Viña del Mar|       1|       1400000.0|       1400000|         22|
|  Valparaíso|       1|       2000000.0|       2000000|         30|
|    Santiago|       2|       2000000.0|       2500000|         25|
+------------+--------+----------------+--------------+-----------+



In [49]:
# Ordenar

print('=== ORDENAR POR SALARIO (descendente) ===')

df_empleados.orderBy(col('salario').desc()).show()



print('\n=== ORDENAR POR EDAD (ascendente) ===')

df_empleados.orderBy('edad').show()

=== ORDENAR POR SALARIO (descendente) ===
+------+----+------------+-------+
|nombre|edad|      ciudad|salario|
+------+----+------------+-------+
|   Ana|  35|    Santiago|2500000|
| María|  30|  Valparaíso|2000000|
| Pedro|  28|  Concepción|1800000|
|  Juan|  25|    Santiago|1500000|
|  Luis|  22|Viña del Mar|1400000|
+------+----+------------+-------+


=== ORDENAR POR EDAD (ascendente) ===
+------+----+------------+-------+
|nombre|edad|      ciudad|salario|
+------+----+------------+-------+
|  Luis|  22|Viña del Mar|1400000|
|  Juan|  25|    Santiago|1500000|
| Pedro|  28|  Concepción|1800000|
| María|  30|  Valparaíso|2000000|
|   Ana|  35|    Santiago|2500000|
+------+----+------------+-------+



## 10.4 SQL con DataFrames



Podemos registrar un DataFrame como una tabla temporal y ejecutar consultas SQL sobre él.

In [50]:
# Registrar DataFrame como tabla temporal

df_empleados.createOrReplaceTempView('empleados')



# Ejecutar consultas SQL

print('=== CONSULTA SQL: SELECT * ===')

spark.sql('SELECT * FROM empleados').show()



print('\n=== CONSULTA SQL: WHERE ===')

spark.sql('SELECT nombre, edad FROM empleados WHERE edad > 25').show()



print('\n=== CONSULTA SQL: GROUP BY ===')

spark.sql('''

    SELECT ciudad,

           COUNT(*) as cantidad,

           AVG(salario) as salario_promedio

    FROM empleados

    GROUP BY ciudad

    ORDER BY salario_promedio DESC

''').show()

=== CONSULTA SQL: SELECT * ===
+------+----+------------+-------+
|nombre|edad|      ciudad|salario|
+------+----+------------+-------+
|  Juan|  25|    Santiago|1500000|
| María|  30|  Valparaíso|2000000|
| Pedro|  28|  Concepción|1800000|
|   Ana|  35|    Santiago|2500000|
|  Luis|  22|Viña del Mar|1400000|
+------+----+------------+-------+


=== CONSULTA SQL: WHERE ===
+------+----+
|nombre|edad|
+------+----+
| María|  30|
| Pedro|  28|
|   Ana|  35|
+------+----+


=== CONSULTA SQL: GROUP BY ===
+------------+--------+----------------+
|      ciudad|cantidad|salario_promedio|
+------------+--------+----------------+
|  Valparaíso|       1|       2000000.0|
|    Santiago|       2|       2000000.0|
|  Concepción|       1|       1800000.0|
|Viña del Mar|       1|       1400000.0|
+------------+--------+----------------+



## 10.5 Convertir entre RDD y DataFrame

In [51]:
# DataFrame a RDD

print('=== DATAFRAME A RDD ===')

rdd_from_df = df_empleados.rdd

print(f'Tipo: {type(rdd_from_df)}')

print('Primeros 2 elementos:')

for row in rdd_from_df.take(2):

    print(f'  {row}')



# RDD a DataFrame

print('\n=== RDD A DATAFRAME ===')

rdd_datos = sc.parallelize([

    ('Laptop', 800000),

    ('Mouse', 15000),

    ('Teclado', 35000)

])



df_from_rdd = spark.createDataFrame(rdd_datos, ['producto', 'precio'])

df_from_rdd.show()

=== DATAFRAME A RDD ===
Tipo: <class 'pyspark.rdd.RDD'>
Primeros 2 elementos:
  Row(nombre='Juan', edad=25, ciudad='Santiago', salario=1500000)
  Row(nombre='María', edad=30, ciudad='Valparaíso', salario=2000000)

=== RDD A DATAFRAME ===
+--------+------+
|producto|precio|
+--------+------+
|  Laptop|800000|
|   Mouse| 15000|
| Teclado| 35000|
+--------+------+



# 11. RESUMEN Y CONCLUSIONES

## Conceptos Clave de RDD y DataFrames

## 11.1 Resumen de Conceptos



### **RDD (Resilient Distributed Datasets)**



✅ **Características fundamentales:**

- Inmutables (no se pueden modificar)

- Distribuidos (particionados en el cluster)

- Resilientes (tolerantes a fallos mediante linaje)

- Evaluación perezosa (lazy evaluation)



✅ **Dos tipos de operaciones:**

- **Transformaciones** (lazy): map, filter, flatMap, reduceByKey, groupByKey, sortByKey, distinct, union

- **Acciones** (eager): count, collect, take, first, reduce, saveAsTextFile



✅ **Evaluación perezosa:**

- Transformaciones construyen un DAG

- Acciones ejecutan el DAG completo

- Permite optimización del plan de ejecución



✅ **Persistencia:**

- `cache()` / `persist()` almacenan RDD en memoria

- Útil cuando un RDD se usa múltiples veces

- `unpersist()` libera la memoria



---



### **DataFrames**



✅ **Ventajas sobre RDDs:**

- API más intuitiva (similar a SQL y Pandas)

- Optimización automática (Catalyst Optimizer)

- Mejor rendimiento (Tungsten engine)

- Esquema estructurado con tipos de datos

- Soporte para múltiples formatos (CSV, JSON, Parquet, JDBC)



✅ **Operaciones comunes:**

- `select()`: Seleccionar columnas

- `filter()` / `where()`: Filtrar filas

- `groupBy()`: Agrupar datos

- `agg()`: Agregaciones (sum, avg, count, max, min)

- `orderBy()` / `sort()`: Ordenar datos

- `withColumn()`: Agregar/modificar columnas



✅ **SQL con DataFrames:**

- `createOrReplaceTempView()`: Registrar como tabla temporal

- `spark.sql()`: Ejecutar consultas SQL



---



### **¿Cuándo usar cada uno?**



| Criterio | RDD | DataFrame |

|----------|-----|----------|

| **Tipo de datos** | No estructurados | Estructurados/Semi-estructurados |

| **Control** | Control fino | API de alto nivel |

| **Rendimiento** | Manual | Optimización automática |

| **API** | Funcional (map, filter) | Declarativa (SQL-like) |

| **Uso recomendado** | Procesamiento personalizado | Análisis de datos, ETL |

## 11.2 Próximos Pasos



En las próximas sesiones del curso profundizaremos en:



1. **Transformaciones avanzadas** en DataFrames

2. **Spark SQL** y consultas complejas

3. **Integración con bases de datos** (MySQL, MongoDB)

4. **Preprocesamiento y limpieza de datos**

5. **Optimización de rendimiento** (particionamiento, broadcast, etc.)

6. **Structured Streaming** para procesamiento en tiempo real



---



## 11.3 Recursos Adicionales



📚 **Documentación oficial:**

- [Apache Spark Documentation](https://spark.apache.org/docs/latest/)

- [PySpark API Reference](https://spark.apache.org/docs/latest/api/python/)



📖 **Libros recomendados:**

- *Learning Spark* (O'Reilly)

- *Spark: The Definitive Guide* (O'Reilly)




## ¡Felicitaciones!



Has completado la cápsula docente de **Introducción a RDD y DataFrames en Apache Spark**.



Ahora tienes los conocimientos fundamentales para trabajar con procesamiento distribuido de datos en Spark. ¡Sigue practicando! 🚀